# Scam checker: train and evaluate

- **Data:** `data/scam_messages.csv`, made from `data/raw/scam_messages.csv` by `scripts/clean_scam_messages.py` (real people's names, a location and a phone number masked; scam links defanged; a non-financial row dropped).
- **Training:** the **synthetic** messages only (189 rows, about 25 templates).
- **Test:** the **real** messages (32 rows: 15 scams, 17 legitimate), never used for training. Only these numbers are reported.
- **Shortcut removed:** most real legitimate SMS contain mask tokens (XXXX) and most scams don't, so the model sees text with masks removed and every number replaced by the word `num`. The rules ignore masks too.
- **Flagged** means the checker says Medium or High. Thresholds (0.40 and 0.70) and the way rules and model are combined were fixed before the real messages were scored.
- **Caveat:** the rules were written after the dataset (including the real messages) had been read, so the rules-only real numbers are optimistic.

In [ ]:
import json
import re
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import pandas as pd
from core.scam_detector import build_model, model_view
from core.scam_training import compare, grouped_cv, load_messages, score_messages

train, test = load_messages(ROOT / "data" / "scam_messages.csv")
print("synthetic (train):", len(train), train["label"].value_counts().to_dict())
print("real (test):", len(test), test["label"].value_counts().to_dict())

In [ ]:
# The shortcut, before and after normalisation: share of messages containing a mask token or a digit.
def share(rows, view):
    return round(rows["text"].map(lambda t: bool(re.search(r"X{2,}|x{3,}|\d", view(t)))).mean(), 2)

for name, part in (("real legitimate", test[test.label == 0]), ("real scam", test[test.label == 1])):
    print(f"{name:16s} raw text: {share(part, lambda t: t):.2f}   model view: {share(part, model_view):.2f}")

In [ ]:
# Development only: synthetic, grouped by template (never reported; the rules were tuned on this set).
print(json.dumps(grouped_cv(train), indent=1))

In [ ]:
# Final model on all synthetic rows; scored once on the real held-out messages.
model = build_model().fit(train["text"], train["label"])
scored = score_messages(test["text"].tolist(), model.predict_proba(test["text"])[:, 1])
real_scores = compare(test["label"].values, scored)
pd.DataFrame(real_scores).T[["recall", "precision", "f1", "accuracy", "tp", "fp", "fn", "tn"]]

In [ ]:
# Every real message: what the checker said (text cut short).
view = pd.concat([test[["label"]].rename(columns={"label": "is_scam"}), scored.round(2), test["text"].str[:80]], axis=1)
view

In [ ]:
joblib.dump(model, ROOT / "models" / "scam_model.joblib", compress=3)
metrics = {
    "real_test": real_scores,
    "flag_threshold": "Medium or High (score >= 0.40)",
    "training_rows": {"synthetic": int(len(train))},
    "notes": "Trained on synthetic messages only; tested on real messages never used for training. "
             "Rules were written after reading the dataset, so rules-only numbers are optimistic.",
}
text = json.dumps(metrics, indent=2)
(ROOT / "models" / "scam_metrics.json").write_text(text + chr(10), encoding="utf-8")
print(text)